# Is the Track A score double-weighting anything?

The score the final 15 were picked by is one line of `examples/track_a_chem_rank.py::rank_weighted`:

$$\mathrm{score} = z(\mathrm{ipsae\_max}) + W \cdot \mathrm{chem\_z}, \qquad \mathrm{chem\_z} = z(\mathrm{contact}) + z(\mathrm{coulomb}), \qquad W = 0.15$$

so its only independent inputs are **ipsae_max, contact and coulomb** (`SCORE_INPUTS`). The other columns of
`ranking.csv` - `ipsae_min`, `iptm`, `in_band`, `n_on_site`, `para_net`, `chem` - are reported and never enter the
number.

Two inputs that measure the same thing would weight that thing twice against the rest, which is why `ipsae_min` is
kept out: it is close to a multiple of `ipsae_max`. This notebook checks every audited column for that pattern on the
committed ranking, and prints what each input is really worth once the inputs correlate.

Pure stdlib, no pandas: the functions come from the ranking script itself, so the notebook and the pipeline check are
the same code.

## Parameters

In [1]:
RANKING = "../data/track_a_final/ranking.csv"   # what `rank --chem-weight 0.15` wrote, one row per candidate
CHEM_WEIGHT = 0.15                              # the weight that ranking was made with
THRESHOLD = 0.9                                 # flag a pair at or above this in |r| or |rho|

## Setup

In [2]:
import csv
import importlib.util
import math
import statistics
import sys
from pathlib import Path

SCRIPT = Path("../examples/track_a_chem_rank.py").resolve()
_spec = importlib.util.spec_from_file_location("track_a_chem_rank", SCRIPT)
chem = importlib.util.module_from_spec(_spec)
sys.modules["track_a_chem_rank"] = chem
_spec.loader.exec_module(chem)

with open(RANKING, newline="") as handle:
    rows = list(csv.DictReader(handle))

print(f"{len(rows)} candidates from {RANKING}")
print("score inputs:", ", ".join(chem.SCORE_INPUTS))
print("audited:     ", ", ".join(chem.AUDIT_COLUMNS))
print("nominal weights:", chem.score_weights(CHEM_WEIGHT))

115 candidates from ../data/track_a_final/ranking.csv
score inputs: ipsae_max, contact, coulomb
audited:      ipsae_max, contact, coulomb, chem_z, chem, ipsae_min, iptm, in_band, n_on_site, para_net
nominal weights: {'ipsae_max': 1.0, 'contact': 0.15, 'coulomb': 0.15}


## 1. ipsae_min against ipsae_max

The regression that got `ipsae_min` excluded: slope, the SD of what is left over, and whether the two columns order
the candidates alike. A residual SD far below `ipsae_min`'s own SD means the column carries almost nothing
`ipsae_max` does not already carry.

In [3]:
pairs = chem._both(rows, "ipsae_max", "ipsae_min")
x = [a for a, _ in pairs]
y = [b for _, b in pairs]
mx, my = statistics.mean(x), statistics.mean(y)
slope = sum((a - mx) * (b - my) for a, b in pairs) / sum((a - mx) ** 2 for a in x)
intercept = my - slope * mx
residuals = [b - (intercept + slope * a) for a, b in pairs]
r = chem.pearson(pairs)

print(f"n                 {len(pairs)}")
print(f"ipsae_min = {intercept:+.4f} + {slope:.4f} * ipsae_max")
print(f"SD(ipsae_min)     {statistics.pstdev(y):.4f}")
print(f"SD(residual)      {statistics.pstdev(residuals):.4f}")
print(f"variance explained {1 - statistics.pvariance(residuals) / statistics.pvariance(y):.3%}")
print(f"pearson r         {r:+.4f}")
print(f"spearman rho      {chem.spearman(pairs):+.4f}")

n                 115
ipsae_min = -0.0218 + 0.8264 * ipsae_max
SD(ipsae_min)     0.1750
SD(residual)      0.0148
variance explained 99.289%
pearson r         +0.9964
spearman rho      +0.9951


## 2. The correlation matrix over every audited column

Pearson first, then Spearman - the one that says whether two columns produce the same ordering. Cells are
pairwise-complete (a candidate with no chemistry still counts in the ipSAE pairs); `.` means the pair had fewer than
three usable rows or a constant column, `*` means it reached the threshold.

In [4]:
for method in ("pearson", "spearman"):
    print(f"--- {method}\n")
    print(chem.format_correlation_matrix(rows, chem.AUDIT_COLUMNS, method, THRESHOLD), end="\n\n")

--- pearson

           ipsae_ma  contact  coulomb   chem_z     chem ipsae_mi     iptm  in_band n_on_sit para_net
ipsae_max     1.000  -0.272   +0.041   -0.121   -0.247   +0.996*  +0.961*  +0.667   +0.781   -0.328 
contact     -0.272     1.000  +0.211   +0.778   +0.799   -0.257   -0.294   -0.188   -0.196   +0.323 
coulomb     +0.041   +0.211     1.000  +0.778   +0.799   +0.047   +0.017   +0.139   +0.153   +0.539 
chem_z      -0.121   +0.778   +0.778     1.000  +1.000*  -0.111   -0.149   -0.031   -0.019   +0.498 
chem        -0.247   +0.799   +0.799   +1.000*    1.000  -0.034   -0.126         .  -0.397   +0.815 
ipsae_min   +0.996*  -0.257   +0.047   -0.111   -0.034     1.000  +0.950*  +0.690   +0.763   -0.331 
iptm        +0.961*  -0.294   +0.017   -0.149   -0.126   +0.950*    1.000  +0.582   +0.730   -0.259 
in_band     +0.667   -0.188   +0.139   -0.031         .  +0.690   +0.582     1.000  +0.457   -0.241 
n_on_site   +0.781   -0.196   +0.153   -0.019   -0.397   +0.763   +0.730   +0.

In [5]:
# every audited pair, worst of |r| and |rho| first, so the matrix above can be read as a list
pear = chem.correlation_matrix(rows, chem.AUDIT_COLUMNS, "pearson")
spear = chem.correlation_matrix(rows, chem.AUDIT_COLUMNS, "spearman")


def worst(pair):
    values = [abs(v) for v in (pear.get(pair, math.nan), spear.get(pair, math.nan)) if not math.isnan(v)]
    return max(values) if values else -1.0


print(f"{'pair':<28}{'r':>8}{'rho':>8}{'n':>5}  note")
for pair in sorted(pear, key=worst, reverse=True):
    a, b = pair
    parents = chem.DERIVED_FROM.get(a, ()) + chem.DERIVED_FROM.get(b, ())
    note = []
    if worst(pair) >= THRESHOLD:
        note.append("OVER")
    if a in parents or b in parents:
        note.append("derived")
    if a in chem.SCORE_INPUTS and b in chem.SCORE_INPUTS:
        note.append("both in score")
    print(f"{a + ' / ' + b:<28}{pear[pair]:+8.3f}{spear.get(pair, math.nan):+8.3f}"
          f"{len(chem._both(rows, a, b)):5d}  {', '.join(note)}")

pair                               r     rho    n  note
chem_z / chem                 +1.000  +1.000   22  OVER
ipsae_max / ipsae_min         +0.996  +0.995  115  OVER
ipsae_max / iptm              +0.961  +0.976  115  OVER
ipsae_min / iptm              +0.950  +0.968  115  OVER
chem / para_net               +0.815  +0.709   22  
contact / chem                +0.799  +0.687   22  derived
coulomb / chem                +0.799  +0.777   22  derived
ipsae_max / n_on_site         +0.781  +0.782  115  
coulomb / chem_z              +0.778  +0.771   97  derived
contact / chem_z              +0.778  +0.746   97  derived
ipsae_min / n_on_site         +0.763  +0.762  115  
iptm / n_on_site              +0.730  +0.761  115  
ipsae_min / in_band           +0.690  +0.681  115  
ipsae_max / in_band           +0.667  +0.681  115  derived
iptm / in_band                +0.582  +0.681  115  
coulomb / para_net            +0.539  +0.511   97  
chem_z / para_net             +0.498  +0.516  115  
in_band /

## 3. What the pipeline check says

`check_score_correlations` is what `rank` now runs on every ranking it writes. Over `SCORE_INPUTS` it is the gate; over
`AUDIT_COLUMNS` it is the wider report, where a pair marked *derived* (`chem`, `chem_z` and `in_band` are functions of
other columns) is expected and is not by itself a reason to drop anything.

In [6]:
print("--- gate: the three columns the score depends on")
gate = chem.check_score_correlations(rows, THRESHOLD, chem.SCORE_INPUTS, stream=sys.stdout)
print("clean" if not gate else f"{len(gate)} pair(s) flagged")

print("\n--- wider report: every audited column")
wide = chem.check_score_correlations(rows, THRESHOLD, chem.AUDIT_COLUMNS, stream=sys.stdout)
print("clean" if not wide else f"{len(wide)} pair(s) flagged")

--- gate: the three columns the score depends on
clean

--- wider report: every audited column
  chem_z and chem: r +1.000, rho +1.000 over 22 candidates
  ipsae_max and ipsae_min: r +0.996, rho +0.995 over 115 candidates
  ipsae_max and iptm: r +0.961, rho +0.976 over 115 candidates
  ipsae_min and iptm: r +0.950, rho +0.968 over 115 candidates
4 pair(s) flagged


## 4. Effective weights, before and after a drop

A nominal weight is not what an input is worth: `score = sum_i w_i z_i`, so `Var(score) = sum_ij w_i w_j r_ij` and the
share input *i* carries is `w_i * sum_j w_j r_ij / Var`. Two correlated inputs lend each other weight. The cell prints
today's shares, then the shares if each input were removed - the "before and after" for any drop under discussion.

In [7]:
before = chem.effective_weights(rows, CHEM_WEIGHT)
print("as scored:")
for name, share in before.items():
    print(f"  {name:<12}{share:8.1%}   nominal {chem.score_weights(CHEM_WEIGHT)[name]:g}")

for column in chem.SCORE_INPUTS:
    after = chem.effective_weights(rows, CHEM_WEIGHT, drop=(column,))
    print(f"\nwithout {column}:")
    for name, share in after.items():
        print(f"  {name:<12}{share:8.1%}   was {before[name]:.1%}")

print("\nipsae_min, iptm, in_band, n_on_site, para_net and chem carry 0% before and after: they are not in the score.")

as scored:
  ipsae_max      98.0%   nominal 1
  contact        -1.4%   nominal 0.15
  coulomb         3.4%   nominal 0.15

without ipsae_max:
  contact        50.0%   was -1.4%
  coulomb        50.0%   was 3.4%

without contact:
  ipsae_max      97.2%   was 98.0%
  coulomb         2.8%   was 3.4%

without coulomb:
  ipsae_max     101.9%   was 98.0%
  contact        -1.9%   was -1.4%

ipsae_min, iptm, in_band, n_on_site, para_net and chem carry 0% before and after: they are not in the score.


## 5. Would a drop change the picks?

The only drop that can move a candidate is a drop of a scored column. This re-scores the committed ranking with one
input removed and reports how far the order moves, so a recommendation can be weighed against what it costs. It
re-ranks the written `score` columns rather than refolding anything.

In [8]:
def rescore(records, weights):
    """Rank the written ranking.csv again under other weights on the same z-scores. A column with no value for a
    candidate contributes 0, as chem_z does in the pipeline."""
    zs = {}
    for column in weights:
        values = {r["name"]: chem._numeric(r.get(column)) for r in records}
        usable = {n: v for n, v in values.items() if not math.isnan(v)}
        mean, sd = statistics.mean(usable.values()), statistics.pstdev(usable.values())
        zs[column] = {n: (v - mean) / (sd if sd else 1.0) for n, v in usable.items()}
    scored = {r["name"]: sum(w * zs[c].get(r["name"], 0.0) for c, w in weights.items()) for r in records}
    return [n for n, _ in sorted(scored.items(), key=lambda kv: -kv[1])]


baseline = rescore(rows, chem.score_weights(CHEM_WEIGHT))
print(f"top 15 as scored: {baseline[:15]}")
for column in chem.SCORE_INPUTS:
    weights = {c: w for c, w in chem.score_weights(CHEM_WEIGHT).items() if c != column}
    order = rescore(rows, weights)
    moved = sum(1 for n in baseline[:15] if n not in order[:15])
    rho = chem.spearman([(float(baseline.index(n)), float(order.index(n))) for n in baseline])
    print(f"without {column:<12} rho(order) {rho:+.4f}   {moved} of the top 15 replaced   new top 15: {order[:15]}")

top 15 as scored: ['k085', 'k063', 'k006', 'k103', 'k105', 'k031', 'k090', 'k056', 'k025', 'k023', 'k061', 'k083', 'k114', 'k045', 'k020']
without ipsae_max    rho(order) +0.0462   13 of the top 15 replaced   new top 15: ['k050', 'k027', 'k006', 'k081', 'k074', 'k073', 'k092', 'k098', 'k030', 'k049', 'k015', 'k090', 'k047', 'k003', 'k086']
without contact      rho(order) +0.9874   1 of the top 15 replaced   new top 15: ['k063', 'k085', 'k105', 'k103', 'k031', 'k056', 'k025', 'k023', 'k061', 'k083', 'k114', 'k006', 'k045', 'k020', 'k012']
without coulomb      rho(order) +0.9889   2 of the top 15 replaced   new top 15: ['k085', 'k103', 'k063', 'k023', 'k105', 'k056', 'k012', 'k090', 'k006', 'k031', 'k061', 'k025', 'k114', 'k083', 'k028']


## How to read this

- A pair over the threshold that is **both in score** is the bug: one quantity weighted twice. Drop the member that is
  the derived or noisier one, keep the one the pipeline and the challenge are stated in terms of.
- A pair over the threshold where one column is **derived** from the other (`chem_z` from `contact` and `coulomb`,
  `in_band` from `ipsae_max`) is arithmetic, not a finding.
- A pair over the threshold where **neither is in the score** costs nothing: both columns are reported only. It is
  still worth knowing, because it says the two columns are not independent evidence when reading the table by eye.
- The sample is 115 candidates from one predictor at three seeds. Seed noise is large (see
  `docs/track_a_final_pipeline.md`), so read a correlation of 0.95 as "these are the same column" and one of 0.6 as
  "related, and the number itself is soft".

## 6. Which columns may carry weight at all

The sections above ask whether two score inputs are secretly the same column.
This asks the question that came before them and was never asked: **is a column
reproducible enough to rank on, and did anything ever test it against labelled
negatives?**

`ipsae_max` carried weight 1.0 in the submitted ranking and fails both halves.
The cells below measure the first half from the per-seed folds already on disk,
read the second half from `03_negative_controls.ipynb`'s verdict table, and
write `qualification.csv` - which `track_a_chem_rank.py` then reads instead of
trusting a tuple edited by hand.

In [9]:
# --- 6. which columns may carry weight: the ICC, and the panel verdict ------
# The audit above asks whether two score inputs are the same column. This asks
# the prior question, which nothing asked before the submission went out: is a
# column reproducible enough to rank on at all, and has anything tested it?
#
# ICC(1) is the share of one fold's variance that belongs to the design rather
# than to the seed. A column at 0.3 means two folds of the same design differ
# about as much as two different designs, so ordering candidates by it is
# ordering them by the random seed. The per-seed values are already on disk in
# `rows_*.csv` - the same folds the submitted ranking was built from.
ROWS_GLOB = "../data/track_a_final/rows_*.csv"
VERDICT = "../runs/negative_controls/verdict.csv"
QUALIFICATION_OUT = "../data/track_a_final/qualification.csv"
ICC_USABLE = 0.75        # at or above: reproducible enough to weight
ICC_MARGINAL = 0.50      # between: reported, not weighted, and said out loud

import glob
from collections import defaultdict

per_seed = defaultdict(lambda: defaultdict(dict))
PER_FOLD_COLUMNS = ("ipsae_max", "ipsae_min", "iptm", "contact", "coulomb",
                    "epitope_overlap", "paratope_net_charge")
for path in sorted(glob.glob(ROWS_GLOB)):
    with open(path, newline="") as handle:
        for row in csv.DictReader(handle):
            for column in PER_FOLD_COLUMNS:
                try:
                    per_seed[column][row["name"]][row["seed"]] = float(row[column])
                except (KeyError, TypeError, ValueError):
                    pass
seed_count = max(
    (len(d) for table in per_seed.values() for d in table.values()), default=0
)
print(f"{len(per_seed)} per-fold columns over {seed_count} seeds, "
      f"from {len(glob.glob(ROWS_GLOB))} rows files")


def icc_one(table, seeds, recurse=True):
    """ICC(1) and its parts, over the designs folded at every seed.

    One-way random effects: between-design variance over between plus within.
    Returns None when fewer than three designs have the full set, because an
    ICC over two designs is not a number anyone should act on.
    """
    names = [n for n, d in table.items() if len(d) == seeds]
    if len(names) < 3 or seeds < 2:
        return None
    grand = statistics.mean(v for n in names for v in table[n].values())
    between = seeds * sum(
        (statistics.mean(table[n].values()) - grand) ** 2 for n in names
    ) / (len(names) - 1)
    within = sum(
        (v - statistics.mean(table[n].values())) ** 2
        for n in names for v in table[n].values()
    ) / (len(names) * (seeds - 1))
    design = max(0.0, (between - within) / seeds)
    if design + within <= 0:
        return None
    constant = [n for n in names
                if max(table[n].values()) == min(table[n].values())]
    varying = {n: table[n] for n in names
               if max(table[n].values()) != min(table[n].values())}
    # Not recursive past one level: the varying subset of a varying subset is
    # itself, so recursing again never terminates.
    inner = (icc_one(varying, seeds, recurse=False)
             if recurse and len(varying) >= 3 else None)
    return {
        "n": len(names),
        "icc": design / (design + within),
        "sd_design": math.sqrt(design),
        "sd_seed": math.sqrt(within),
        "se_mean": math.sqrt(within / seeds),
        "constant": len(constant),
        "varying": len(varying),
        "icc_varying": inner["icc"] if inner else float("nan"),
    }


measured = {}
for column in PER_FOLD_COLUMNS:
    out = icc_one(per_seed[column], seed_count)
    if out is not None:
        measured[column] = out

print()
print(f"{'column':<22}{'n':>5}{'ICC(1)':>8}{'sd design':>11}{'sd seed':>9}"
      f"{'SE of mean':>12}{'constant':>10}{'ICC varying':>13}")
print("-" * 92)
for column, out in measured.items():
    constant = f"{out['constant']}/{out['n']}"
    inner = ("-" if out["icc_varying"] != out["icc_varying"]
             else f"{out['icc_varying']:.3f}")
    print(f"{column:<22}{out['n']:>5}{out['icc']:>8.3f}{out['sd_design']:>11.4f}"
          f"{out['sd_seed']:>9.4f}{out['se_mean']:>12.4f}{constant:>10}{inner:>13}")
print("-" * 92)
print("'constant' is how many designs give the identical value at every seed, and")
print("'ICC varying' is the ICC among the rest. The two together catch a column")
print("whose headline ICC is carried by a mass of identical values rather than by")
print("any reproducible ordering - which is exactly what `contact` turns out to be.")


7 per-fold columns over 3 seeds, from 8 rows files

column                    n  ICC(1)  sd design  sd seed  SE of mean  constant  ICC varying
--------------------------------------------------------------------------------------------
ipsae_max               116   0.298     0.1591   0.2442      0.1410     0/116        0.298
ipsae_min               116   0.306     0.1332   0.2005      0.1157     0/116        0.306
iptm                    116   0.278     0.1084   0.1747      0.1008     0/116        0.278
contact                 115   0.398     0.2637   0.3244      0.1873    71/115        0.000
coulomb                 115   0.734     0.1119   0.0674      0.0389     1/115        0.733
epitope_overlap         115   0.336     0.1666   0.2343      0.1353     1/115        0.311
paratope_net_charge     115   0.464     1.0489   1.1280      0.6513    10/115        0.409
--------------------------------------------------------------------------------------------
'constant' is how many designs giv

In [10]:
# --- the qualification table the ranker reads -------------------------------
# One row per column that could enter a score. `qualified` is a conjunction of
# two independent requirements, and a column has to meet both: it must have
# been tested against labelled negatives, and it must be reproducible across
# seeds. Either one alone is how `ipsae_max` came to carry weight 1.0.
verdict_rows = {}
if Path(VERDICT).is_file():
    with open(VERDICT, newline="") as handle:
        verdict_rows = {r["model"]: r for r in csv.DictReader(handle)}
    print(f"panel verdicts for {len(verdict_rows)} scorers from {VERDICT}")
else:
    print(f"no {VERDICT}; run 03_negative_controls.ipynb first. Without it no")
    print("column can be marked qualified, which is the safe direction to fail in.")

# Which panel column decides which scorer, and what the scorer is called in
# `ranking.csv`. ipSAE is the odd one: the panel knows it as `ipsae` and the
# ranking as `ipsae_max`.
PANEL_NAME = {"ipsae_max": "ipsae", "foldx": "foldx"}

# Columns that are not computed from a fold, so there is no seed for an ICC to
# be about. FoldX reads one fixed repaired structure; its own reproducibility is
# the run-to-run spread of `--numberOfRuns`, which `_foldx_score.py` averages
# and does not retain, so it is not measurable from what is on disk either. The
# exemption is recorded in the table rather than left implicit, because "we did
# not measure this" and "this has no seed" are different claims.
NOT_FOLD_DERIVED = {
    "foldx": "not a fold-derived column: one fixed repaired structure, no Boltz "
             "seed. Its run-to-run spread is averaged away by FOLDX_RUNS and not "
             "retained, so no ICC is computable from these files",
}


def panel_verdict(column):
    row = verdict_rows.get(PANEL_NAME.get(column, column))
    if row is None:
        return None, "not in the panel"
    call = str(row.get("affinity_verdict", "")).strip()
    auroc, low, high = (row.get("direction_auroc"), row.get("direction_ci_low"),
                        row.get("direction_ci_high"))
    try:
        detail = (f"direction AUROC {float(auroc):.3f} "
                  f"(CI {float(low):.2f}-{float(high):.2f})")
    except (TypeError, ValueError):
        detail = "direction not measured"
    return call, detail


qualification = []
for column in sorted(set(PER_FOLD_COLUMNS) | set(PANEL_NAME) | {"chem"}):
    out = measured.get(column)
    icc = out["icc"] if out else float("nan")
    icc_varying = out["icc_varying"] if out else float("nan")
    # A column whose varying subset has no reproducible component fails on the
    # inner number whatever the headline says.
    effective = (icc_varying if (icc_varying == icc_varying and out
                                 and out["varying"] >= 3 and icc_varying < icc)
                 else icc)
    call, detail = panel_verdict(column)
    reasons = []
    if call == "pass":
        tested = True
    elif call is None:
        tested = False
        reasons.append("no panel test")
    else:
        tested = False
        reasons.append(f"panel says {call} ({detail})")
    if column in NOT_FOLD_DERIVED:
        reproducible = True
        reasons.append(NOT_FOLD_DERIVED[column])
    elif effective != effective:
        reproducible = False
        reasons.append("no per-seed values, so no ICC")
    elif effective >= ICC_USABLE:
        reproducible = True
    else:
        reproducible = False
        band = "marginal" if effective >= ICC_MARGINAL else "mostly seed noise"
        reasons.append(f"ICC {effective:.3f} ({band})")
    qualification.append({
        "column": column,
        "qualified": bool(tested and reproducible),
        "tested_on_negatives": tested,
        "panel_verdict": call or "",
        "panel_detail": detail if call else "",
        "icc": "" if icc != icc else round(icc, 4),
        "icc_varying": "" if icc_varying != icc_varying else round(icc_varying, 4),
        "icc_used": "" if effective != effective else round(effective, 4),
        "n_designs": out["n"] if out else "",
        "seeds": seed_count if out else "",
        "why": "; ".join(reasons) if reasons else "panel pass and ICC at or above "
               f"{ICC_USABLE}",
    })

Path(QUALIFICATION_OUT).parent.mkdir(parents=True, exist_ok=True)
with open(QUALIFICATION_OUT, "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(qualification[0]))
    writer.writeheader()
    writer.writerows(qualification)

print()
print(f"{'column':<22}{'qualified':>11}{'tested':>8}{'ICC used':>10}   why")
print("-" * 104)
for row in qualification:
    print(f"{row['column']:<22}{str(row['qualified']):>11}"
          f"{str(row['tested_on_negatives']):>8}{str(row['icc_used']):>10}   {row['why']}")
print("-" * 104)
print(f"-> {QUALIFICATION_OUT}")
print()
print("`examples/track_a_chem_rank.py rank --qualification` reads this file and")
print("refuses to call a ranking qualified if any column it weights is not")
print("cleared here. That is the check whose absence let an untested column")
print("carry the whole score.")


panel verdicts for 9 scorers from ../runs/negative_controls/verdict.csv

column                  qualified  tested  ICC used   why
--------------------------------------------------------------------------------------------------------
chem                        False   False             no panel test; no per-seed values, so no ICC
contact                     False   False       0.0   no panel test; ICC 0.000 (mostly seed noise)
coulomb                     False   False    0.7331   no panel test; ICC 0.733 (marginal)
epitope_overlap             False   False    0.3113   no panel test; ICC 0.311 (mostly seed noise)
foldx                        True    True             not a fold-derived column: one fixed repaired structure, no Boltz seed. Its run-to-run spread is averaged away by FOLDX_RUNS and not retained, so no ICC is computable from these files
ipsae_max                   False   False    0.2982   no panel test; ICC 0.298 (mostly seed noise)
ipsae_min                   False   Fals